In [13]:
import json
from pprint import pprint

from __future__ import annotations

# Import typing helpers for type hints 
# These help define what type each field should be 
from typing import Dict, List, Literal, Optional

# BaseModel is the core class used to define structured data models 
from pydantic import BaseModel, Field, ConfigDict, model_validator, ValidationError

# **<u>2. TED-style Speechwriter/Editor + Structure Validator Agent</u>**
- ted_speechwriter
- ted_validator

**<u>High-level Overview</u>** 

User Input -> **Planner Agent** <-> Planner Schema Validation (Pydantic) -> **TED Agent** <-> TED Schema Validation (Pydantic) -> **Structure Agent** <-> Structure Schema Validation (Pydantic) -> **Content Agent** <-> Content Schema Validation (Pydantic) -> Grounding Agent -> **Style Agent** <-> **Reflection Agent** -> **Judging Agent** -> Final Response or go back to any agents.

**A) What the TED agent expects as <u>input</u>**

Define a minimal "PlannerBlueprint" schema that includes only is truly need, e.g. 
- `topic`
- `audience`
- `time_limit_minutes`
- `required_points[]`
- `sections[]` (title, purpose, key_points, word_budget)

Even if the planner later adds more fields, your TED agent can ignore extras.

**B) What the TED agent <u>outputs</u>**

Define a TEDBlueprint schema that your checker can validate: 
- `opening_hook`
- `big_idea`
- `narrative_arc`
- `sections[]` with `spoken_beats[]`, `transition_in/out`, `word_budget`
- `retrieval_requests[]` (placeholders only; no facts yet)

**2) Create a mock planner output (so it can develop now)**
Make a `mock_planner_blueprint.json` with 2-4 sections. Example: 
- Section 1: Definition / context 
- Section 2: Use case #1 
- Section 3: Use case #2 
- Section 4: Wrap-up 

Then we can: 
- Run TED agent end-to-end
- Run the checker
- Test the "reject -> loop back to TED agent" logic 

## **2.1. Planner Agent**
---
- Output: `mock_planner_blueprint.json`

In [5]:
# Open the mock planner blueprint JSON file 
with open("mocks/mock_planner_blueprint.json", "r") as f:
    mock_planner_blueprint = json.load(f)

pprint(mock_planner_blueprint, sort_dicts=False)

{'blueprint_version': '1.0',
 'request': {'topic': 'How Artificial Intelligence is Transforming '
                      'Manufacturing Excellence',
             'audience': 'Global manufacturing leaders, plant managers, and '
                         'engineering teams',
             'occasion': 'BMW Group Global Manufacturing Summit 2026 – Opening '
                         'Keynote',
             'time_limit_minutes': 6,
             'style_target': 'BMW',
             'style_strictness': 'medium'},
 'targets': {'estimated_wpm': 140,
             'target_word_count': 840,
             'tone_keywords': ['confident', 'structured', 'forward-looking']},
 'constraints': {'required_points': [{'id': 'RP1',
                                      'text': 'Explain what AI means in the '
                                              'context of smart manufacturing'},
                                     {'id': 'RP2',
                                      'text': 'Provide 2 concrete use cases '
 

## **2.2. Planner Schema Validation (Pydantic)**
---
- Input: `mock_planner_blueprint.json`
- Validation: `planner_blueprint.py`
- Output: `mock_planner_blueprint.json` 

In [6]:
# Define allowed values for voice strictness 
# Literal ensures only these values are allowed 
StyleStrictness = Literal["low", "medium", "high"]

# ---------------------------------------------------
# REQUEST SECTION
# ---------------------------------------------------
# Contains the original request information from the user
class Request(BaseModel):
    topic: str = Field(..., description="Main topic of the speech")
    audience: str = Field(..., description="Target audience")
    occasion: str = Field(..., description="Event where the speech will be delivered")
    time_limit_minutes: int = Field(
        ..., # required field 
        ge=1, # must be >=1 
        le=60, # must be <= 60
        description="Maximum allowed speech duration"
    )
    style_target: str = Field(
        default="BMW",
        description="Corporate voice to emulate"
    )
    style_strictness: StyleStrictness=Field(
        default="medium",
        description="How strictly the style should be followed"
    )

# ---------------------------------------------------
# TARGETS SECTION
# ---------------------------------------------------
# Defines speech length and tone 
class Targets(BaseModel):
    estimated_wpm: int = Field(
        default=140,
        description="Estimated speaking speed"
    )
    target_word_count: int = Field(
        ...,
        description="Target speech length in words"
    )
    tone_keywords: List[str] = Field(
        default_factory=list,
        description="Tone guidance keywords"
    )

# ---------------------------------------------------
# REQUIRED POINTS
# ---------------------------------------------------
# Required points that must appear somewhere in the speech 
class RequiredPoint(BaseModel):
    id: str = Field(
        ...,
        pattern=r"^RP\d+$", # Must look like RP1, RP2, RP3 
        description="Unique identifier for required point"
    )
    text: str = Field(
        ...,
        description="Text describing the required point"
    )

# ---------------------------------------------------
# CONSTRAINTS SECTION
# ---------------------------------------------------
class Constraints(BaseModel):
    required_points: List[RequiredPoint] = Field(
        default_factory=list 
    )
    avoid_topics: List[str] = Field(
        default_factory=list
    )
    max_iterations: int = Field(
        default=2,
        description="Maximum allowed iteration loops"
    )

# ---------------------------------------------------
# OUTLINE SECTION
# ---------------------------------------------------
# Represents each planned section of the speech 
class OutlineSection(BaseModel):
    id: str = Field(
        ...,
        pattern=r"^S\d+$", # Must look like S1, S2, S3
        description="Section identifier"
    )
    label: str = Field(
        ...,
        description="Name of the section"
    )
    purpose: str = Field(
        ...,
        description="Purpose of the section"
    )
    key_points: List[str] = Field(
        default_factory=list,
        description="Key talking points"
    )
    must_include: List[str] = Field(
        default_factory=list,
        description="Points that must appear inside the section"
    )
    word_budget: int = Field(
        ...,
        ge=0,
        description="Maximum words allowed for section"
    )

# ---------------------------------------------------
# COVERAGE MAP
# ---------------------------------------------------
class CoverageMap(BaseModel):
    required_points_to_sections: Dict[str, List[str]] = Field(
        default_factory=dict,
        description="Mapping from required point ID -> section IDs"
    )

# ---------------------------------------------------
# MAIN BLUEPRINT MODEL
# ---------------------------------------------------
# This is the top-level schema used by the planner 
class PlannerBlueprint(BaseModel):
    
    model_config = ConfigDict(extra="forbid") # Config ensures no unexpected keys appear 
    blueprint_version: str = Field(default="1.0")
    request: Request # Request metadata 
    targets: Targets # Speech targets 
    constraints: Constraints # Planner constraints 
    outline: List[OutlineSection] # Outline sections
    coverage_map: CoverageMap # Coverage mapping 

    # ---------------------------------------------------
    # POST-VALIDATION CHECK
    # ---------------------------------------------------
    # Runs after model validation 
    @model_validator(mode="after")
    def check_internal_consistency(self):

        # Collect section IDs
        section_ids = [section.id for section in self.outline]

        # Ensure section IDs are unique 
        if len(section_ids) != len(set(section_ids)):
            raise ValueError("Duplicate section ID detected")
        
        return self 

In [ ]:
# How this will be used in LangGraph 
from schemas.planner_blueprint import PlannerBlueprint

planner_blueprint = PlannerBlueprint.model_validate(mock_planner_blueprint)
# Then your TED agent receives a guaranteed structure 

# Note: Once architecture is up, directly validate from planner agent output: 
# PlannerBlueprint.model_validate_json(planner_blueprint)

In [9]:
# Think about what if the validation fails, what is the next step? 
# - Ask user to edit? 

## **2.3. TED Agent**
It is a speech structure optimizer. It does not add facts, do web search, generate final speech text, or change required points. It ONLY transforms a planner outline into a TED-style narrative structure.

Planner outline (logical strcuture) -> Speech blueprint (narrative + spoken structure)

Planner Output (example): 
- S1 Opening
- S2 Definition of AI in Smart Manufacturing 
- S3 Automotive Use Cases
- S4 Implications + Close 

TED Blueprint (added rhetorical structure, spoken beats, transitions, but still no actual speech text yet): 
- Opening Hook
- Big Idea
- Problem Tension
- Breakthrough Insight 
- Evidence / Examples 
- Takeaway 
---
- Input: `mock_planner_blueprint.json` 
- Validation: `ted_blueprint.py`
- Output: `ted_blueprint.json`

In [20]:
# Suggested system prompt 
system_prompt = """
You are a TED-style speech structure editor.

Your task is to transform a planner blueprint into a TED-style speech blueprint optimized for spoken delivery.

The planner blueprint contains logical sections and constraints. Your job is to refine it into a narrative structure suitable for a TED-style talk.

Important rules:

1. Do NOT generate the final speech text.
2. Do NOT invent statistics, facts, or citations.
3. Preserve all required points from the planner blueprint.
4. Maintain the original meaning and intent of the planner outline.
5. Convert logical sections into narrative sections suitable for spoken delivery.
6. Add a strong opening hook.
7. Define a single clear "big idea".
8. Convert bullet points into spoken beats (short speech ideas).
9. Add transitions between sections where appropriate.
10. If examples or statistics are needed, create retrieval requests instead of fabricating information.

TED-style narrative structure typically follows:

Hook → Context/Challenge → Insight → Evidence/Examples → Takeaway

Return output ONLY in the TED blueprint JSON schema provided.
Do not include explanations, markdown, or commentary.
"""

user_prompt = """
Transform the following planner blueprint into a TED-style speech blueprint.

Planner Blueprint:
{planner_blueprint_json}

Instructions:
1. Preserve the request, targets, constraints, and coverage_map fields.
2. Keep the original planner outline as "original_outline".
3. Add the following fields:
   * hook
   * big_idea
   * narrative_arc
   * ted_sections
   * retrieval_requests
4. Convert the planner outline sections into TED narrative sections.
5. Assign a narrative role to each section (hook, insight, example, takeaway, etc.).
6. Convert planner key_points into spoken_beats suitable for spoken delivery.
7. Add transitions between sections where appropriate.

Return ONLY valid JSON following the TEDBlueprint schema.
"""

In [19]:
# Open the mock ted blueprint JSON file 
with open("mocks/mock_ted_blueprint.json", "r") as f:
    mock_ted_blueprint = json.load(f)

pprint(mock_ted_blueprint, sort_dicts=False)

{'blueprint_version': '1.0',
 'source_blueprint_type': 'planner_blueprint',
 'blueprint_type': 'ted_blueprint',
 'request': {'topic': 'How Artificial Intelligence is Transforming '
                      'Manufacturing Excellence',
             'audience': 'Global manufacturing leaders, plant managers, and '
                         'engineering teams',
             'occasion': 'BMW Group Global Manufacturing Summit 2026 – Opening '
                         'Keynote',
             'time_limit_minutes': 6,
             'style_target': 'BMW',
             'style_strictness': 'medium'},
 'targets': {'estimated_wpm': 140,
             'target_word_count': 840,
             'tone_keywords': ['confident', 'structured', 'forward-looking']},
 'constraints': {'required_points': [{'id': 'RP1',
                                      'text': 'Explain what AI means in the '
                                              'context of smart manufacturing'},
                                     {'id': 'RP